# NeurIPS Ariel Data Challenge 2025 — Stage 1: Ingest & CV Design
**Track A (Classical Physics-Informed ML)**  
**Scope:** Ingest metadata, verify contract shapes, extract ADC parameters, inspect `axis_info`, confirm 1:1 star-planet mapping, assign spectral classes (M/K/G/F), freeze 5-fold `StratifiedGroupKFold` CV with a 10% holdout vault, verify metric self-test, inspect label bounds, and confirm raw sensor cube dimensions on one target.

In [1]:
VERSION = "1.1.0"
SEED = 42

import os
import sys
import hashlib
from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from sklearn.model_selection import StratifiedGroupKFold

print(f"Notebook Version: {VERSION} | Seed: {SEED}")
print(f"Python:   {sys.version.split()[0]}")
print(f"NumPy:    {np.__version__}")
print(f"Pandas:   {pd.__version__}")
print(f"PyArrow:  {pa.__version__}")

Notebook Version: 1.1.0 | Seed: 42
Python:   3.12.13
NumPy:    2.0.2
Pandas:   2.3.3
PyArrow:  24.0.0


In [2]:
def resolve_data_paths():
    candidates = [
        Path("/kaggle/input/competitions/ariel-data-challenge-2025"),
        Path("/kaggle/input/ariel-data-challenge-2025"),
        Path("./data/raw"),
        Path("../data/raw")
    ]
    for cand in candidates:
        if cand.exists() and (cand / "train_star_info.csv").exists():
            return cand
    for root in [Path("/kaggle/input"), Path(".")]:
        if root.exists():
            for dirpath, _, filenames in os.walk(root):
                if "train_star_info.csv" in filenames:
                    return Path(dirpath)
    raise FileNotFoundError("Unable to locate competition data directory containing train_star_info.csv")

DATA_DIR = resolve_data_paths()
OUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./data/cache")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"DATA_DIR: {DATA_DIR}")
print(f"OUT_DIR:  {OUT_DIR}")

DATA_DIR: /kaggle/input/competitions/ariel-data-challenge-2025
OUT_DIR:  /kaggle/working


In [3]:
star_df = pd.read_csv(DATA_DIR / "train_star_info.csv")
labels_df = pd.read_csv(DATA_DIR / "train.csv")
adc_df = pd.read_csv(DATA_DIR / "adc_info.csv")
axis_df = pd.read_parquet(DATA_DIR / "axis_info.parquet")
sample_sub = pd.read_csv(DATA_DIR / "sample_submission.csv")

# Contract shape assertions
assert labels_df.shape == (1100, 284), f"labels_df shape mismatch: expected (1100, 284), got {labels_df.shape}"
assert sample_sub.shape[1] == 567, f"sample_submission.csv must have 567 columns (1 ID + 283 mu + 283 sigma), got {sample_sub.shape[1]}"
assert "planet_id" in sample_sub.columns, "sample_submission.csv missing planet_id!"

print(f"train_star_info:   {star_df.shape}")
print(f"train labels:      {labels_df.shape} (asserted 1100 x 284)")
print(f"axis_info:         {axis_df.shape}")
print(f"sample_submission: {sample_sub.shape} (asserted 567 columns)")

# Strict ADC key extraction
required_adc_cols = ["FGS1_adc_gain", "FGS1_adc_offset", "AIRS-CH0_adc_gain", "AIRS-CH0_adc_offset"]
for col in required_adc_cols:
    assert col in adc_df.columns, f"Required ADC column '{col}' missing from adc_info.csv!"

FGS1_ADC_GAIN = float(adc_df["FGS1_adc_gain"].iloc[0])
FGS1_ADC_OFFSET = float(adc_df["FGS1_adc_offset"].iloc[0])
AIRS_ADC_GAIN = float(adc_df["AIRS-CH0_adc_gain"].iloc[0])
AIRS_ADC_OFFSET = float(adc_df["AIRS-CH0_adc_offset"].iloc[0])

print("\nExtracted ADC Constants:")
print(f"  FGS1 ADC Gain:       {FGS1_ADC_GAIN:.4f} e-/ADU | Offset: {FGS1_ADC_OFFSET:.1f} ADU")
print(f"  AIRS-CH0 ADC Gain:   {AIRS_ADC_GAIN:.4f} e-/ADU | Offset: {AIRS_ADC_OFFSET:.1f} ADU")

train_star_info:   (1100, 9)
train labels:      (1100, 284) (asserted 1100 x 284)
axis_info:         (135000, 4)
sample_submission: (1, 567) (asserted 567 columns)

Extracted ADC Constants:
  FGS1 ADC Gain:       0.4369 e-/ADU | Offset: -1000.0 ADU
  AIRS-CH0 ADC Gain:   0.4369 e-/ADU | Offset: -1000.0 ADU


In [4]:
wl_col = axis_df["AIRS-CH0-axis2-um"].dropna()
wl_min = float(wl_col.min())
wl_max = float(wl_col.max())
is_increasing = bool(wl_col.is_monotonic_increasing)
is_decreasing = bool(wl_col.is_monotonic_decreasing)

print("=" * 65)
print("AXIS_INFO WAVELENGTH GRID SUMMARY:")
print(f"  Raw Spectral Bins:   {len(wl_col)}")
print(f"  Wavelength Min:      {wl_min:.5f} um")
print(f"  Wavelength Max:      {wl_max:.5f} um")
print(f"  Monotonic Increasing:{is_increasing}")
print(f"  Monotonic Decreasing:{is_decreasing}")
print("  Note: Crop window [37:319] and dispersion axis orientation are not frozen here; deferred to Stage 3.")
print("=" * 65)

AXIS_INFO WAVELENGTH GRID SUMMARY:
  Raw Spectral Bins:   356
  Wavelength Min:      1.63325 um
  Wavelength Max:      4.07846 um
  Monotonic Increasing:False
  Monotonic Decreasing:True
  Note: Crop window [37:319] and dispersion axis orientation are not frozen here; deferred to Stage 3.


In [5]:
# 1:1 star-planet check
n_planets = len(star_df)
n_unique_planets = star_df["planet_id"].nunique()
n_unique_stars = star_df[["Ts", "Rs", "Ms"]].drop_duplicates().shape[0]

assert n_planets == n_unique_planets == n_unique_stars, "Detected multi-planet systems or duplicate stars!"
print(f"✓ 1:1 host star to planet correspondence confirmed: {n_planets} unique systems.")

star_df["star_id"] = star_df["planet_id"].astype(str)

# Surface gravity log10(g) in cgs: log10(g_sun) ~ 4.4378
star_df["logg"] = 4.4378 + np.log10(star_df["Ms"] / (star_df["Rs"] ** 2))

# Astrophysical spectral class assignment
def get_spectral_class(ts):
    if ts < 3700.0: return "M"
    elif ts < 5200.0: return "K"
    elif ts < 6000.0: return "G"
    else: return "F"

star_df["spectral_class"] = star_df["Ts"].apply(get_spectral_class)
print("\nSpectral class distribution:")
print(star_df["spectral_class"].value_counts().to_dict())

✓ 1:1 host star to planet correspondence confirmed: 1100 unique systems.

Spectral class distribution:
{'G': 477, 'F': 474, 'K': 149}


In [6]:
# 1. Carve out a 10% stratified holdout vault (~112 planets)
rng = np.random.default_rng(SEED)
vault_indices = []
for s_class, group in star_df.groupby("spectral_class"):
    idx = group.index.values.copy()
    rng.shuffle(idx)
    n_vault = int(np.round(len(idx) * 0.102))
    vault_indices.extend(idx[:n_vault])

star_df["is_vault"] = False
star_df.loc[vault_indices, "is_vault"] = True
non_vault_df = star_df[~star_df["is_vault"]].copy()

# 2. 5-fold StratifiedGroupKFold on remaining non-vault stars
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
non_vault_df["fold"] = -1

for fold_id, (train_idx, val_idx) in enumerate(sgkf.split(
    X=non_vault_df,
    y=non_vault_df["spectral_class"],
    groups=non_vault_df["star_id"]
)):
    val_planet_ids = non_vault_df.iloc[val_idx]["planet_id"]
    non_vault_df.loc[non_vault_df["planet_id"].isin(val_planet_ids), "fold"] = fold_id

star_df["fold"] = -1
star_df.loc[non_vault_df.index, "fold"] = non_vault_df["fold"]

# 3. Zero-leakage assertion check
for f in range(5):
    p_f = set(star_df[star_df["fold"] == f]["planet_id"])
    for f_other in range(f + 1, 5):
        p_other = set(star_df[star_df["fold"] == f_other]["planet_id"])
        assert len(p_f.intersection(p_other)) == 0, f"Leakage between fold {f} and {f_other}!"

vault_planets = set(star_df[star_df["is_vault"]]["planet_id"])
for f in range(5):
    p_f = set(star_df[star_df["fold"] == f]["planet_id"])
    assert len(vault_planets.intersection(p_f)) == 0, f"Vault leaked into fold {f}!"

print(f"✓ Zero leakage confirmed across 5 folds and {len(vault_planets)} vault planets (10.2%).")
display(pd.crosstab(star_df["fold"], star_df["spectral_class"]))

✓ Zero leakage confirmed across 5 folds and 112 vault planets (10.2%).


spectral_class,F,G,K
fold,,,
-1,48,49,15
0,86,92,20
1,83,84,31
2,87,79,31
3,86,88,24
4,84,85,28


In [7]:
FGS1_WEIGHT = 57.846
AIRS_WEIGHT = 1.0
NUM_CHANNELS = 283

def get_official_channel_weights():
    w = np.ones(NUM_CHANNELS, dtype=np.float64) * AIRS_WEIGHT
    w[0] = FGS1_WEIGHT
    return w

def compute_gll(y_true, mu_pred, sigma_pred):
    weights = get_official_channel_weights()
    y_true = np.asarray(y_true, dtype=np.float64)
    mu_pred = np.asarray(mu_pred, dtype=np.float64)
    sigma_pred = np.asarray(sigma_pred, dtype=np.float64)
    
    if np.any(np.isnan(y_true)) or np.any(np.isnan(mu_pred)) or np.any(np.isnan(sigma_pred)):
        raise ValueError("NaN values detected in inputs or predictions!")
    if np.any(np.isinf(y_true)) or np.any(np.isinf(mu_pred)) or np.any(np.isinf(sigma_pred)):
        raise ValueError("Infinite values detected in inputs or predictions!")
    if np.any(sigma_pred <= 0):
        raise ValueError("sigma_pred must be strictly positive (sigma > 0)!")
        
    var = sigma_pred ** 2
    gll_per_channel = -0.5 * (np.log(2.0 * np.pi * var) + ((y_true - mu_pred) ** 2) / var)
    return np.sum(weights * gll_per_channel, axis=-1) / np.sum(weights)

# Self-test: FGS1 error penalty ratio
y_mock = np.full((1, NUM_CHANNELS), 0.012)
sig_mock = np.full((1, NUM_CHANNELS), 0.001)
base = compute_gll(y_mock, y_mock, sig_mock)[0]

mu_fgs = y_mock.copy(); mu_fgs[0, 0] += 0.001
mu_airs = y_mock.copy(); mu_airs[0, 1] += 0.001

ratio = (base - compute_gll(y_mock, mu_fgs, sig_mock)[0]) / (base - compute_gll(y_mock, mu_airs, sig_mock)[0])
assert np.isclose(ratio, FGS1_WEIGHT, atol=1e-3)
print(f"✓ GLL metric verified. FGS1 penalty ratio: {ratio:.4f} (expected {FGS1_WEIGHT:.4f})")

✓ GLL metric verified. FGS1 penalty ratio: 57.8460 (expected 57.8460)


In [8]:
num_cols = [c for c in labels_df.columns if c != "planet_id"]
fgs1_vals = labels_df[num_cols[0]].values
airs_vals = labels_df[num_cols[1:]].values

print("=" * 65)
print("GROUND TRUTH TRANSMISSION SPECTRUM SCALE (Rp/Rs)^2:")
print("=" * 65)
print(f"FGS1 (Channel 0):")
print(f"  Mean:   {np.mean(fgs1_vals):.6f} ({np.mean(fgs1_vals)*1e6:.1f} ppm)")
print(f"  Std:    {np.std(fgs1_vals):.6f}")
print(f"  Min:    {np.min(fgs1_vals):.6f}")
print(f"  Max:    {np.max(fgs1_vals):.6f}")
print(f"\nAIRS-CH0 (Channels 1-282):")
print(f"  Mean:   {np.mean(airs_vals):.6f} ({np.mean(airs_vals)*1e6:.1f} ppm)")
print(f"  Std:    {np.std(airs_vals):.6f}")
print(f"  Min:    {np.min(airs_vals):.6f}")
print(f"  Max:    {np.max(airs_vals):.6f}")

assert np.all(fgs1_vals >= 0.0) and np.all(airs_vals >= 0.0), "Negative depths found!"
assert np.all(fgs1_vals <= 0.08), f"FGS1 transit depth exceeds 8%: {np.max(fgs1_vals)}"
assert np.all(airs_vals <= 0.10), f"AIRS transit depth exceeds physical bound of 10%: {np.max(airs_vals)}"
if np.max(airs_vals) > 0.08:
    print(f"  Notice: Maximum AIRS channel depth reaches {np.max(airs_vals):.6f} in deep molecular features (within <= 0.10 physical bound).")
print("✓ Physical bounds verified: all depths lie strictly within valid exoplanetary range.")

GROUND TRUTH TRANSMISSION SPECTRUM SCALE (Rp/Rs)^2:
FGS1 (Channel 0):
  Mean:   0.014609 (14609.3 ppm)
  Std:    0.010647
  Min:    0.003654
  Max:    0.079797

AIRS-CH0 (Channels 1-282):
  Mean:   0.014689 (14689.3 ppm)
  Std:    0.010661
  Min:    0.003662
  Max:    0.088650
  Notice: Maximum AIRS channel depth reaches 0.088650 in deep molecular features (within <= 0.10 physical bound).
✓ Physical bounds verified: all depths lie strictly within valid exoplanetary range.


In [9]:
train_dir = DATA_DIR / "train"
sample_target = sorted([p for p in train_dir.iterdir() if p.is_dir()])[0]

print(f"Target Planet: {sample_target.name}")

# Visit discovery helper
fgs_signals = sorted(sample_target.glob("FGS1_signal_*.parquet"))
airs_signals = sorted(sample_target.glob("AIRS-CH0_signal_*.parquet"))
print(f"\nVisit Discovery on Target {sample_target.name}:")
print(f"  FGS1 signal files ({len(fgs_signals)}):     {[f.name for f in fgs_signals]}")
print(f"  AIRS-CH0 signal files ({len(airs_signals)}): {[f.name for f in airs_signals]}")
if len(fgs_signals) > 1:
    print("  -> Multi-visit structure detected on disk for this target.")
else:
    print("  -> Single-visit structure on disk for this target.")

# Load raw parquet signals and verify tensor shapes
fgs_raw = pq.read_table(sample_target / "FGS1_signal_0.parquet").to_pandas().values
airs_raw = pq.read_table(sample_target / "AIRS-CH0_signal_0.parquet").to_pandas().values

print(f"\nRaw FGS1 shape:     {fgs_raw.shape}, dtype: {fgs_raw.dtype}")
print(f"Raw AIRS-CH0 shape: {airs_raw.shape}, dtype: {airs_raw.dtype}")

# Verify 3D tensor reshape (no differencing or calibration applied in Stage 1)
fgs_cube = fgs_raw.reshape(-1, 32, 32)
airs_cube = airs_raw.reshape(-1, 32, 356)

print(f"Reshaped FGS1 cube:     {fgs_cube.shape}")
print(f"Reshaped AIRS-CH0 cube: {airs_cube.shape}")

Target Planet: 1010375142

Visit Discovery on Target 1010375142:
  FGS1 signal files (1):     ['FGS1_signal_0.parquet']
  AIRS-CH0 signal files (1): ['AIRS-CH0_signal_0.parquet']
  -> Single-visit structure on disk for this target.

Raw FGS1 shape:     (135000, 1024), dtype: uint16
Raw AIRS-CH0 shape: (11250, 11392), dtype: uint16
Reshaped FGS1 cube:     (135000, 32, 32)
Reshaped AIRS-CH0 cube: (11250, 32, 356)


In [10]:
manifest_cols = [
    "star_id", "planet_id", "spectral_class", "Ts", "Rs", "Ms", "logg", "Mp", "e", "P", "sma", "i", "fold", "is_vault"
]
manifest_export = star_df[manifest_cols].copy()

# Pre-save manifest schema checks
for col in manifest_cols:
    assert col in manifest_export.columns, f"Missing manifest column: {col}"
assert set(manifest_export["fold"].unique()).issubset({-1, 0, 1, 2, 3, 4}), f"Invalid folds: {manifest_export['fold'].unique()}"
assert np.isclose(manifest_export["is_vault"].mean(), 0.102, atol=0.015), f"Vault fraction anomaly: {manifest_export['is_vault'].mean()}"

manifest_path = OUT_DIR / "C01_manifest.csv"
manifest_export.to_csv(manifest_path, index=False)

cv_export = star_df[["star_id", "planet_id", "spectral_class", "fold", "is_vault"]].copy()
cv_path = OUT_DIR / "C02_cv_folds.csv"
cv_export.to_csv(cv_path, index=False)

vault_export = star_df[star_df["is_vault"]][["star_id", "planet_id", "spectral_class"]].copy()
vault_path = OUT_DIR / "vault_planets.csv"
vault_export.to_csv(vault_path, index=False)

def sha256(filepath):
    h = hashlib.sha256()
    with open(filepath, "rb") as f:
        while chunk := f.read(65536):
            h.update(chunk)
    return h.hexdigest()

print("=" * 70)
print("FROZEN STAGE 1 ARTIFACTS & SHA-256 HASHES:")
print("=" * 70)
for p in [manifest_path, cv_path, vault_path]:
    print(f"  {p.name:20s}: {sha256(p)}")
print("=" * 70)

FROZEN STAGE 1 ARTIFACTS & SHA-256 HASHES:
  C01_manifest.csv    : 904f828233dac80faab1b45863b0ac6daf8b109aa5217ac3a0f59470237f4931
  C02_cv_folds.csv    : 749ba0175458d12ee004397f234be2d5fdae72e40479bbae99cac79f414c8c86
  vault_planets.csv   : 6356583ef846309d2c114857c2919bb46fecce75a21525e665810574749b6312


Stage 1 complete. Artifacts frozen. Proceed to Stage 2 for FGS1 detector calibration.